# GrC Image Matting on CBIS-DDSM - Layers 2-4 (v2)

Six-channel template matching, histogram pooling, K selection and SVM tuning on the **validation** split, and fast Layer 4 probability maps.

## State restoration

In [1]:
# Imports and config
!pip install -q opencv-python scikit-learn scipy tqdm joblib

import os, json, random, warnings, time, gc
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
import joblib
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive/cbis_ddsm_grc_project_v2"
CONFIG = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
globals().update(CONFIG)
SEED = CONFIG["SEED"]
random.seed(SEED); np.random.seed(SEED)

MAX_PER_CLASS_PER_IMG = 300        # sampled definite-mass / definite-background pixels per image
MAX_FIT = 20000                    # SVM training rows
MAX_TUNE_FIT = 8000                # SVM training rows while comparing K and (C, gamma)
INTENSITY_SCALES = [5, 9, 15]
print("Config loaded.")


Mounted at /content/drive
Config loaded.


In [2]:
final_df_loaded = pd.read_csv(os.path.join(RESULTS_DIR, "final_df_loaded.csv"))
metadata_df     = pd.read_csv(os.path.join(PREPROCESSED_DIR, "preprocessing_metadata.csv"))
roi_arch    = np.load(os.path.join(PREPROCESSED_DIR, "rois.npz"))
trimap_arch = np.load(os.path.join(TRIMAPS_DIR, "trimaps.npz"))

class_of = dict(zip(final_df_loaded['image_id'], final_df_loaded['class']))
preprocessed_results = {}
for iid in roi_arch.files:
    preprocessed_results[iid] = {
        'roi': roi_arch[iid],
        'trimap': trimap_arch[iid] if iid in trimap_arch.files else None,
        'class': class_of.get(iid)}

# ---- patient-grouped split (train / validation / test) ----
sp = json.load(open(os.path.join(RESULTS_DIR, "train_test_split.json")))
train_ids, val_ids, test_ids = set(sp['train_ids']), set(sp['val_ids']), set(sp['test_ids'])
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    for unit in ('lesions', 'patients'):
        assert not (set(sp[f'{a}_{unit}']) & set(sp[f'{b}_{unit}'])), f"{unit} leakage {a}/{b}"
print(f"Split: train {len(train_ids)}  val {len(val_ids)}  test {len(test_ids)} images - no lesion/patient overlap.")

# ---- six Layer 1 channels (LIPW and LBPW at 3, 5, 7) and their TRAIN-only statistics ----
CHANNELS = [(ps, kind) for ps in PATCH_SIZES for kind in ('lipw', 'lbpw')]
layer1_maps = {ps: {} for ps in PATCH_SIZES}
for ps in PATCH_SIZES:
    for kind in ('lipw', 'lbpw'):
        a = np.load(os.path.join(OUTPUTS_DIR, f"layer1_ms_{kind}_{ps}x{ps}.npz"))
        for iid in a.files:
            layer1_maps[ps].setdefault(iid, {})[kind] = a[iid]
cs = json.load(open(os.path.join(MODELS_DIR, "channel_stats.json")))
assert [tuple(c) for c in cs['channels']] == CHANNELS, "channel order differs from Notebook 2"
CH_MEAN, CH_STD = np.array(cs['mean'], np.float32), np.array(cs['std'], np.float32)

# ---- template bank ----
tmeta = json.load(open(os.path.join(MODELS_DIR, "template_meta.json")))
K_CANDIDATES = tmeta['k_candidates']
CENTER_MODE  = tmeta['center_mode']
bank = {k: np.load(os.path.join(MODELS_DIR, f"templates_K{k}.npy")) for k in K_CANDIDATES}
N_CH, WIN = len(CHANNELS), tmeta['window']
assert all(t.shape == (k, N_CH, WIN, WIN) for k, t in bank.items())
print(f"Restored {len(preprocessed_results)} images; bank K={K_CANDIDATES}; centre mode '{CENTER_MODE}'.")


Split: train 642  val 156  test 269 images - no lesion/patient overlap.
Restored 1067 images; bank K=[8, 12, 16]; centre mode 'channel'.


## Layer 2, Layer 3 and feature functions

In [3]:
def build_stack(iid):
    """H x W x 6 float32 stack of the six Layer 1 maps, z-scored with train statistics."""
    st = np.stack([layer1_maps[ps][iid][kind] for ps, kind in CHANNELS], axis=2).astype(np.float32)
    return (st - CH_MEAN[None, None, :]) / CH_STD[None, None, :]

def window_view(stack, ps):
    half = ps // 2
    padded = np.pad(stack, ((half, half), (half, half), (0, 0)), mode='reflect')
    return sliding_window_view(padded, (ps, ps), axis=(0, 1))

def center_patches(P, mode='channel'):
    P = P.reshape(len(P), N_CH, -1)
    if mode == 'channel':
        P = P - P.mean(axis=2, keepdims=True)
    elif mode == 'joint':
        P = P - P.reshape(len(P), -1).mean(axis=1)[:, None, None]
    return P.reshape(len(P), -1)

def prepare_signs(templates, mode=CENTER_MODE):
    """Sign patterns (the only part of a template the matcher uses), computed once per K."""
    Tc = center_patches(templates.reshape(len(templates), -1).astype(np.float32), mode)
    W = np.sign(Tc); W[W == 0] = 1.0
    return W, np.linalg.norm(W, axis=1)

def match_templates_signed_stack(stack, W, Wn, mode=CENTER_MODE):
    """H x W x K scores in [0, 1] (signed cosine, Eqs. 5-6)."""
    h, w, C = stack.shape
    pf = window_view(stack, WIN).reshape(h * w, C * WIN * WIN).astype(np.float32)
    Pc = center_patches(pf, mode)
    num = Pc @ W.T
    den = np.linalg.norm(Pc, axis=1, keepdims=True) * Wn[None, :] + 1e-8
    return (((num / den) + 1.0) * 0.5).reshape(h, w, len(W)).astype(np.float32)

def compute_histogram_map_fast(score_map, window_size=7):
    """Layer 3: normalised local histogram of template scores (box-filter pooling)."""
    h, w, k = score_map.shape
    area = float(window_size * window_size)
    summed = np.empty_like(score_map, dtype=np.float32)
    for c in range(k):
        summed[:, :, c] = cv2.boxFilter(score_map[:, :, c], ddepth=-1, ksize=(window_size, window_size),
                                        borderType=cv2.BORDER_REFLECT, normalize=True) * area
    totals = summed.sum(axis=2, keepdims=True)
    return np.where(totals > 1e-8, summed / totals, 1.0 / k).astype(np.float32)

def local_intensity_maps(roi, scales=INTENSITY_SCALES):
    return [cv2.boxFilter(roi.astype(np.float32), -1, (w, w), borderType=cv2.BORDER_REFLECT) for w in scales]

def layer3_for(iid, W, Wn):
    return compute_histogram_map_fast(match_templates_signed_stack(build_stack(iid), W, Wn), HISTOGRAM_WINDOW)

def aligned(iid, hh, ww):
    tm, roi = preprocessed_results[iid]['trimap'], preprocessed_results[iid]['roi']
    if tm.shape != (hh, ww):  tm  = cv2.resize(tm,  (ww, hh), interpolation=cv2.INTER_NEAREST)
    if roi.shape != (hh, ww): roi = cv2.resize(roi, (ww, hh), interpolation=cv2.INTER_LINEAR)
    return tm, roi

SIGNS = {k: prepare_signs(t) for k, t in bank.items()}

def build_feature_tables(ids, ks, max_per_class=MAX_PER_CLASS_PER_IMG):
    """One pass over the images. The SAME pixels are sampled for every K, so K candidates are compared
    on identical rows. Returns {k: X (n x (k+3))}, y, groups, and the intensity-only block."""
    rng = np.random.default_rng(SEED)
    Xk, Xi, y, grp = {k: [] for k in ks}, [], [], []
    for iid in tqdm(sorted(ids), desc="features"):
        if iid not in preprocessed_results or preprocessed_results[iid]['trimap'] is None \
                or iid not in layer1_maps[PATCH_SIZES[0]]:
            continue
        st_hw = layer1_maps[PATCH_SIZES[0]][iid]['lipw'].shape
        tm, roi = aligned(iid, *st_hw)
        pm, pb = np.argwhere(tm == 255), np.argwhere(tm == 0)
        if len(pm) == 0 or len(pb) == 0:
            continue
        n = min(len(pm), len(pb), max_per_class)
        sel = []
        for pos, lab in ((pm, 1), (pb, 0)):
            if len(pos) > n:
                pos = pos[rng.choice(len(pos), n, replace=False)]
            sel.append((pos, lab))
        pos_all = np.concatenate([p for p, _ in sel])
        lab_all = np.concatenate([np.full(len(p), l) for p, l in sel])
        ims = local_intensity_maps(roi)
        Xi.append(np.stack([im[pos_all[:, 0], pos_all[:, 1]] for im in ims], axis=1))
        for k in ks:
            hm = layer3_for(iid, *SIGNS[k])
            Xk[k].append(hm[pos_all[:, 0], pos_all[:, 1]])
        y.append(lab_all); grp += [iid] * len(pos_all)
    Xi = np.concatenate(Xi).astype(np.float32)
    return ({k: np.concatenate([np.concatenate(Xk[k]), Xi], axis=1).astype(np.float32) for k in ks},
            Xi, np.concatenate(y), np.array(grp))

print("Functions ready.")


Functions ready.


## K selection on the validation split

In [4]:
t0 = time.time()
tables_tr, Xi_tr, y_tr, g_tr = build_feature_tables(train_ids, K_CANDIDATES)
tables_va, Xi_va, y_va, g_va = build_feature_tables(val_ids,   K_CANDIDATES)
print(f"Feature tables built in {(time.time()-t0)/60:.1f} min | train {len(y_tr)} rows, val {len(y_va)} rows "
      f"| balanced: train mass {y_tr.mean():.2f}, val mass {y_va.mean():.2f}")

def quick_fit_score(Xtr, ytr, Xva, yva, C=10, gamma='scale', n_fit=MAX_TUNE_FIT, seed=SEED):
    sc = StandardScaler().fit(Xtr)
    idx = np.random.default_rng(seed).choice(len(Xtr), min(n_fit, len(Xtr)), replace=False)
    m = SVC(kernel='rbf', C=C, gamma=gamma, class_weight='balanced', probability=False, random_state=seed)
    m.fit(sc.transform(Xtr[idx]), ytr[idx])
    return balanced_accuracy_score(yva, m.predict(sc.transform(Xva)))

K_VAL = {'intensity only': quick_fit_score(Xi_tr, y_tr, Xi_va, y_va)}
for k in K_CANDIDATES:
    K_VAL[k] = quick_fit_score(tables_tr[k], y_tr, tables_va[k], y_va)

print(f"\n{'features':<18}{'val balanced acc':>18}")
print("-" * 36)
for name, v in K_VAL.items():
    print(f"{('K = ' + str(name) if name != 'intensity only' else name):<18}{v:>18.4f}")

best = max(K_VAL[k] for k in K_CANDIDATES)
FINAL_K = min(k for k in K_CANDIDATES if K_VAL[k] >= best - 0.01)
print(f"\nBest K score {best:.4f}; smallest K within 0.01 of it: K = {FINAL_K}")
print(f"Gain of texture bins over intensity-only at K={FINAL_K}: {K_VAL[FINAL_K] - K_VAL['intensity only']:+.4f}")
print("(differences of about 0.01 or less are within noise on this split; the tie rule exists for that reason)")


features: 100%|██████████| 156/156 [00:17<00:00,  8.93it/s]


Feature tables built in 1.7 min | train 219930 rows, val 57102 rows | balanced: train mass 0.50, val mass 0.50

features            val balanced acc
------------------------------------
intensity only                0.5435
K = 8                         0.5365
K = 12                        0.5377
K = 16                        0.5353

Best K score 0.5377; smallest K within 0.01 of it: K = 8
Gain of texture bins over intensity-only at K=8: -0.0070
(differences of about 0.01 or less are within noise on this split; the tie rule exists for that reason)


## SVM hyper-parameters on the validation split

In [5]:
Xtr, Xva = tables_tr[FINAL_K], tables_va[FINAL_K]
print(f"Selected K = {FINAL_K}; feature width {Xtr.shape[1]} ({FINAL_K} bins + {len(INTENSITY_SCALES)} intensity scales)")
assert Xtr.shape[1] == FINAL_K + len(INTENSITY_SCALES)

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)
rng = np.random.default_rng(SEED)
gi = rng.choice(len(Xtr_s), min(MAX_TUNE_FIT, len(Xtr_s)), replace=False)

grid_rows = []
for C in SVM_C_VALUES:
    for gam in SVM_GAMMA_VALUES:
        m = SVC(kernel='rbf', C=C, gamma=gam, class_weight='balanced', probability=False, random_state=SEED)
        m.fit(Xtr_s[gi], y_tr[gi])
        grid_rows.append((C, gam, balanced_accuracy_score(y_va, m.predict(Xva_s))))
print(f"{'C':>6} {'gamma':>8} {'val bal. acc':>14}")
for C, gam, s in grid_rows:
    print(f"{C:>6} {gam:>8} {s:>14.4f}")
best_s = max(r[2] for r in grid_rows)
BEST_C, BEST_GAMMA = sorted([r for r in grid_rows if r[2] >= best_s - 1e-12], key=lambda r: (r[0], str(r[1])))[0][:2]
print(f"\nBest on validation: C={BEST_C}, gamma={BEST_GAMMA} ({best_s:.4f})")

fi = rng.choice(len(Xtr_s), min(MAX_FIT, len(Xtr_s)), replace=False)
svm = SVC(kernel='rbf', C=BEST_C, gamma=BEST_GAMMA, class_weight='balanced',
          probability=True, random_state=SEED)
svm.fit(Xtr_s[fi], y_tr[fi])
va_pred = svm.predict(Xva_s)
val_acc, val_bal = accuracy_score(y_va, va_pred), balanced_accuracy_score(y_va, va_pred)
print(f"\nValidation accuracy {val_acc:.4f} | balanced {val_bal:.4f} | support vectors {len(svm.support_vectors_)}")
print(classification_report(y_va, va_pred, target_names=['background', 'mass'], digits=4))

json.dump({'selected_K': int(FINAL_K), 'k_validation_scores': {str(k): float(v) for k, v in K_VAL.items()},
           'selection_rule': 'smallest K within 0.01 of the best validation balanced accuracy'},
          open(os.path.join(MODELS_DIR, "selected_K.json"), 'w'), indent=2)
joblib.dump(svm, os.path.join(MODELS_DIR, "svm_layer4.joblib"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "svm_scaler.joblib"))
json.dump({'selected_K': int(FINAL_K), 'best_params': {'C': BEST_C, 'gamma': str(BEST_GAMMA)},
           'val_balanced_acc_tuning': float(best_s), 'val_acc': float(val_acc), 'val_balanced_acc': float(val_bal),
           'intensity_only_val_balanced_acc': float(K_VAL['intensity only']),
           'n_fit_rows': int(len(fi)), 'n_val_rows': int(len(y_va)), 'n_support_vectors': int(len(svm.support_vectors_)),
           'test_used_for_fitting_or_selection': False},
          open(os.path.join(RESULTS_DIR, "svm_results.json"), 'w'), indent=2)
print("Saved selected_K.json, SVM, scaler, svm_results.json")


Selected K = 8; feature width 11 (8 bins + 3 intensity scales)
     C    gamma   val bal. acc
   0.1    scale         0.5464
   0.1     auto         0.5465
     1    scale         0.5452
     1     auto         0.5453
    10    scale         0.5365
    10     auto         0.5365

Best on validation: C=0.1, gamma=auto (0.5465)

Validation accuracy 0.5446 | balanced 0.5446 | support vectors 18506
              precision    recall  f1-score   support

  background     0.5990    0.2700    0.3723     28551
        mass     0.5288    0.8192    0.6427     28551

    accuracy                         0.5446     57102
   macro avg     0.5639    0.5446    0.5075     57102
weighted avg     0.5639    0.5446    0.5075     57102

Saved selected_K.json, SVM, scaler, svm_results.json


## Fast Layer 4 prediction

In [6]:
try:
    import torch
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    torch, DEV = None, 'numpy'

class FastRBFProba:
    """P(mass) from a fitted RBF SVC: kernel via matrix products + sklearn's Platt parameters."""
    def __init__(self, svm, scaler, chunk=4096):
        self.mu, self.sd = scaler.mean_.astype(np.float64), scaler.scale_.astype(np.float64)
        self.sv = svm.support_vectors_.astype(np.float64)
        self.coef, self.b = svm.dual_coef_[0].astype(np.float64), float(svm.intercept_[0])
        self.gamma = float(svm._gamma)
        self.A, self.B = float(svm.probA_[0]), float(svm.probB_[0])
        self.chunk = chunk
        self.sv_sq = (self.sv ** 2).sum(1)
        self.sv32, self.sv_sq32, self.coef32 = (self.sv.astype(np.float32), self.sv_sq.astype(np.float32),
                                                 self.coef.astype(np.float32))
        if DEV != 'numpy':
            dt = torch.float32
            self.t_sv = torch.tensor(self.sv, dtype=dt, device=DEV)
            self.t_coef = torch.tensor(self.coef, dtype=dt, device=DEV)
            self.t_sq = torch.tensor(self.sv_sq, dtype=dt, device=DEV)
    def decision(self, Xraw):
        out = np.empty(len(Xraw), np.float64)
        for s in range(0, len(Xraw), self.chunk):
            Z = (Xraw[s:s + self.chunk].astype(np.float64) - self.mu) / self.sd
            if DEV == 'numpy':
                Z = Z.astype(np.float32)
                d2 = (Z ** 2).sum(1)[:, None] + self.sv_sq32[None, :] - 2.0 * Z @ self.sv32.T
                np.maximum(d2, 0, out=d2); d2 *= -np.float32(self.gamma); np.exp(d2, out=d2)
                out[s:s + self.chunk] = (d2 @ self.coef32).astype(np.float64) + self.b
            else:
                Zt = torch.tensor(Z, dtype=torch.float32, device=DEV)
                d2 = (Zt ** 2).sum(1, keepdim=True) + self.t_sq[None, :] - 2.0 * Zt @ self.t_sv.T
                out[s:s + self.chunk] = (torch.exp(-self.gamma * d2.clamp_min(0)) @ self.t_coef).double().cpu().numpy() + self.b
        return out
    def proba(self, Xraw):
        f = self.decision(Xraw)
        return (1.0 / (1.0 + np.exp(self.A * f - self.B))).astype(np.float32)   # sklearn sign convention

def pixel_features(iid, hist_map):
    hh, ww, _ = hist_map.shape
    _, roi = aligned(iid, hh, ww)
    ims = local_intensity_maps(roi)
    return np.concatenate([hist_map.reshape(-1, hist_map.shape[2]),
                           np.stack([im.reshape(-1) for im in ims], axis=1)], axis=1), (hh, ww)

fast = FastRBFProba(svm, scaler)
print(f"Fast predictor on: {DEV}  ({len(fast.sv)} support vectors)")

# ---- verify against sklearn on real validation pixels from 3 images ----
chk_ids = sorted(val_ids)[:3]
Xchk = np.concatenate([pixel_features(i, layer3_for(i, *SIGNS[FINAL_K]))[0][::7] for i in chk_ids])[:5000]
t0 = time.time(); p_ref = svm.predict_proba(scaler.transform(Xchk))[:, 1]; t_ref = time.time() - t0
t0 = time.time(); p_fast = fast.proba(Xchk);                                  t_fast = time.time() - t0
max_err, mean_err = float(np.abs(p_ref - p_fast).max()), float(np.abs(p_ref - p_fast).mean())
agree = float(((p_ref > 0.5) == (p_fast > 0.5)).mean())
USE_FAST = max_err <= 0.01
print(f"{len(Xchk)} pixels | sklearn predict_proba {t_ref:.2f}s vs fast {t_fast:.2f}s ({t_ref / max(t_fast, 1e-9):.1f}x)")
print(f"max |diff| {max_err:.4f}, mean |diff| {mean_err:.5f}, same side of 0.5 for {agree:.4f} of pixels")
print("Using fast predictor." if USE_FAST else "Difference too large - falling back to predict_proba.")

def predict_prob_map(iid, hist_map):
    X, (hh, ww) = pixel_features(iid, hist_map)
    if USE_FAST:
        p = fast.proba(X)
    else:
        p = np.concatenate([svm.predict_proba(scaler.transform(X[s:s + 20000]))[:, 1]
                            for s in range(0, len(X), 20000)])
    return p.reshape(hh, ww).astype(np.float32)


Fast predictor on: cuda  (18506 support vectors)
5000 pixels | sklearn predict_proba 4.64s vs fast 0.34s (13.7x)
max |diff| 0.0050, mean |diff| 0.00073, same side of 0.5 for 0.9846 of pixels
Using fast predictor.


## Layer 4 probability maps (validation and test images)

Inference only. Train images are not needed downstream (the ablation arms retrain their own classifiers in Notebook 4).

In [ ]:
t0 = time.time()
layer4_maps = {}
for iid in tqdm(sorted(val_ids | test_ids), desc="Layer 4"):
    if iid not in preprocessed_results or iid not in layer1_maps[PATCH_SIZES[0]]:
        continue
    layer4_maps[iid] = predict_prob_map(iid, layer3_for(iid, *SIGNS[FINAL_K]))
print(f"{len(layer4_maps)} probability maps in {(time.time()-t0)/60:.1f} min "
      f"({(time.time()-t0)/max(len(layer4_maps),1):.2f} s/image)")
np.savez_compressed(os.path.join(OUTPUTS_DIR, "layer4_prob_maps.npz"), **layer4_maps)
json.dump({'val_ids': sorted(i for i in layer4_maps if i in val_ids),
           'test_ids': sorted(i for i in layer4_maps if i in test_ids), 'selected_K': int(FINAL_K)},
          open(os.path.join(RESULTS_DIR, "layer4_map_ids.json"), 'w'))
print("Saved layer4_prob_maps.npz")

Layer 4: 100%|██████████| 425/425 [00:44<00:00,  9.64it/s]


425 probability maps in 0.7 min (0.10 s/image)
Saved layer4_prob_maps.npz


## Feature diagnostic 

Per-dimension class difference of the histogram bins vs the intensity scales, and two classifier-free / alternative-classifier checks on the **validation** split.

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
mass_mean, bg_mean = Xtr[y_tr == 1].mean(axis=0), Xtr[y_tr == 0].mean(axis=0)
diff = mass_mean - bg_mean
print(f"{'dim':<10}{'mass mean':>11}{'bg mean':>10}{'diff':>9}")
print("-" * 40)
for i in range(len(diff)):
    name = f"T{i}" if i < FINAL_K else f"I{INTENSITY_SCALES[i - FINAL_K]}x{INTENSITY_SCALES[i - FINAL_K]}"
    print(f"{name:<10}{mass_mean[i]:>11.4f}{bg_mean[i]:>10.4f}{diff[i]:>+9.4f}")
print(f"\nmax |diff| over histogram bins : {np.abs(diff[:FINAL_K]).max():.4f}  (uniform bin mass is {1.0 / FINAL_K:.4f})")
print(f"max |diff| over intensity dims : {np.abs(diff[FINAL_K:]).max():.4f}")

d_m = np.linalg.norm(Xva - mass_mean, axis=1); d_b = np.linalg.norm(Xva - bg_mean, axis=1)
nc_acc = float(((d_m < d_b).astype(int) == y_va).mean())
gb = HistGradientBoostingClassifier(random_state=SEED, class_weight='balanced').fit(Xtr[fi], y_tr[fi])
gb_bal = balanced_accuracy_score(y_va, gb.predict(Xva))
print(f"\nValidation balanced accuracy | nearest centroid {nc_acc:.4f} | gradient boosting {gb_bal:.4f} "
      f"| SVM {val_bal:.4f} | intensity-only SVM {K_VAL['intensity only']:.4f}")
print("If very different classifiers land in the same band, that points to a feature ceiling rather than a classifier problem.")


dim         mass mean   bg mean     diff
----------------------------------------
T0             0.1244    0.1247  -0.0003
T1             0.1246    0.1249  -0.0002
T2             0.1255    0.1254  +0.0001
T3             0.1253    0.1251  +0.0001
T4             0.1252    0.1250  +0.0003
T5             0.1254    0.1251  +0.0003
T6             0.1247    0.1249  -0.0002
T7             0.1249    0.1249  -0.0001
I5x5           0.6129    0.5793  +0.0336
I9x9           0.6125    0.5793  +0.0332
I15x15         0.6116    0.5792  +0.0324

max |diff| over histogram bins : 0.0003  (uniform bin mass is 0.1250)
max |diff| over intensity dims : 0.0336

Validation balanced accuracy | nearest centroid 0.5254 | gradient boosting 0.5340 | SVM 0.5446 | intensity-only SVM 0.5435
If very different classifiers land in the same band, that points to a feature ceiling rather than a classifier problem.


## Verification

In [9]:
def verify_notebook3():
    spj = json.load(open(os.path.join(RESULTS_DIR, "svm_results.json")))
    checks = {
        "K selected on validation, in bank"        : FINAL_K in K_CANDIDATES,
        "feature width = K + 3"                    : Xtr.shape[1] == FINAL_K + len(INTENSITY_SCALES),
        "SVM trained on train rows only"           : not (set(g_tr) & (val_ids | test_ids)),
        "validation rows disjoint from train rows" : not (set(g_va) & train_ids),
        "no test image used for fitting/selection" : spj['test_used_for_fitting_or_selection'] is False,
        "validation acc above chance"              : val_bal > 0.5,
        "prob maps for (nearly) every val+test image": len(layer4_maps) >= 0.99 * len(val_ids | test_ids),
        "prob in [0,1]"                            : all(m.min() >= 0 and m.max() <= 1 for m in layer4_maps.values()),
        "fast predictor agrees with sklearn"       : (not USE_FAST) or max_err <= 0.01,
        "saved to Drive"                           : all(os.path.exists(p) for p in [
            os.path.join(OUTPUTS_DIR, "layer4_prob_maps.npz"), os.path.join(MODELS_DIR, "svm_layer4.joblib"),
            os.path.join(MODELS_DIR, "selected_K.json")]),
    }
    print("=" * 56); print("NOTEBOOK 3 VERIFICATION"); print("=" * 56)
    ok = True
    for k, v in checks.items():
        print(f"  {'OK  ' if v else 'FAIL'}  {k}"); ok = ok and v
    print("=" * 56); print("Proceed to Notebook 4." if ok else "Review failures.")
    return ok

verify_notebook3()


NOTEBOOK 3 VERIFICATION
  OK    K selected on validation, in bank
  OK    feature width = K + 3
  OK    SVM trained on train rows only
  OK    validation rows disjoint from train rows
  OK    no test image used for fitting/selection
  OK    validation acc above chance
  OK    prob maps for (nearly) every val+test image
  OK    prob in [0,1]
  OK    fast predictor agrees with sklearn
  OK    saved to Drive
Proceed to Notebook 4.


True